In [6]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import yfinance as yf
from datetime import datetime, timedelta
import os
import warnings
import csv
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD AND PARSE CSV DATA
# ============================================================================

csv_path = r"c:\Users\soman\OneDrive\Coding\SOC 2026\option-chain-ED-NIFTY-09-Jun-2026.csv"

# Read the CSV file using proper CSV parsing
with open(csv_path, 'r') as f:
    reader = csv.reader(f)
    all_rows = list(reader)

# Parse the data - skip first 2 header lines
data = []
for row in all_rows[2:]:  # Skip "CALLS,,PUTS" and header row
    if len(row) >= 23:
        # Structure: empty | call(0-9) | strike | put(0-9) | ...
        call_data = row[1:11]   # 10 call columns
        strike = row[11]        # Strike price
        put_data = row[12:22]   # 10 put columns
        data.append(call_data + [strike] + put_data)

# Create DataFrame with proper column names
call_cols = ['CALL_OI', 'CALL_CHNG_OI', 'CALL_VOLUME', 'CALL_IV', 'CALL_LTP', 'CALL_CHNG', 'CALL_BID_QTY', 'CALL_BID', 'CALL_ASK', 'CALL_ASK_QTY']
strike_col = ['STRIKE']
put_cols = ['PUT_BID_QTY', 'PUT_BID', 'PUT_ASK', 'PUT_ASK_QTY', 'PUT_CHNG', 'PUT_LTP', 'PUT_IV', 'PUT_VOLUME', 'PUT_CHNG_OI', 'PUT_OI']

columns = call_cols + strike_col + put_cols

df = pd.DataFrame(data, columns=columns)

# Clean up data - convert to numeric where applicable, removing commas and dashes
def clean_numeric(val):
    if pd.isna(val) or val == '' or val == '-':
        return np.nan
    val = str(val).replace(',', '').strip()
    try:
        return float(val)
    except:
        return np.nan

for col in df.columns:
    df[col] = df[col].apply(clean_numeric)

# Remove rows with NaN strikes
df = df.dropna(subset=['STRIKE'])
df = df.reset_index(drop=True)

print("✓ Option Chain Data Loaded Successfully!")
print(f"✓ Total strikes available: {len(df)}")
print(f"\nStrike range: {df['STRIKE'].min():,.0f} to {df['STRIKE'].max():,.0f}")
print(f"\nSample data (ATM strikes around 23,350-23,400):")
atm_range = df[(df['STRIKE'] >= 23300) & (df['STRIKE'] <= 23450)].copy()
print(atm_range[['STRIKE', 'CALL_LTP', 'CALL_IV', 'PUT_LTP', 'PUT_IV']].to_string())

✓ Option Chain Data Loaded Successfully!
✓ Total strikes available: 101

Strike range: 21,100 to 26,100

Sample data (ATM strikes around 23,350-23,400):
     STRIKE  CALL_LTP  CALL_IV  PUT_LTP  PUT_IV
44  23300.0     190.0    14.28    84.70   12.88
45  23350.0     161.0    14.24   105.70   12.89
46  23400.0     134.7    14.19   129.45   12.87
47  23450.0     111.8    14.20   156.05   12.81


In [7]:
# ============================================================================
# 2. STRIKE SELECTION & ATM IDENTIFICATION
# ============================================================================

S = 23366.70  # Spot price
snapshot_date = datetime(2026, 6, 5)
expiry_date = datetime(2026, 6, 9)
T_days = 4
T = T_days / 365  # Time to maturity in years
r = 0.065  # Risk-free rate (6.5%)

# Find the closest ATM strike
df['distance_to_spot'] = abs(df['STRIKE'] - S)
atm_idx = df['distance_to_spot'].idxmin()
atm_strike = df.loc[atm_idx]

K = atm_strike['STRIKE']
CALL_LTP_MKT = atm_strike['CALL_LTP']
CALL_IV_MKT = atm_strike['CALL_IV'] / 100  # Convert to decimal
PUT_LTP_MKT = atm_strike['PUT_LTP']
PUT_IV_MKT = atm_strike['PUT_IV'] / 100  # Convert to decimal

print("=" * 70)
print("STRIKE SELECTION & PARAMETERS")
print("=" * 70)
print(f"\nSpot Price (S):                    {S:,.2f}")
print(f"Selected ATM Strike (K):           {K:,.2f}")
print(f"Distance from Spot:                {abs(K - S):,.2f} points")
print(f"\nSnapshot Date:                     {snapshot_date.strftime('%Y-%m-%d')}")
print(f"Expiry Date:                       {expiry_date.strftime('%Y-%m-%d')}")
print(f"Time to Maturity (T):              {T_days} days = {T:.6f} years")
print(f"Risk-Free Rate (r):                {r*100:.2f}%")

print(f"\n{'Metric':<30} {'Call':<15} {'Put':<15}")
print("-" * 60)
print(f"{'Market LTP':<30} {CALL_LTP_MKT:<15.2f} {PUT_LTP_MKT:<15.2f}")
print(f"{'Market IV (%)':<30} {CALL_IV_MKT*100:<15.2f} {PUT_IV_MKT*100:<15.2f}")

STRIKE SELECTION & PARAMETERS

Spot Price (S):                    23,366.70
Selected ATM Strike (K):           23,350.00
Distance from Spot:                16.70 points

Snapshot Date:                     2026-06-05
Expiry Date:                       2026-06-09
Time to Maturity (T):              4 days = 0.010959 years
Risk-Free Rate (r):                6.50%

Metric                         Call            Put            
------------------------------------------------------------
Market LTP                     161.00          105.70         
Market IV (%)                  14.24           12.89          


In [9]:
# ============================================================================
# 3. HISTORICAL VOLATILITY CALCULATION
# ============================================================================

print("\n" + "=" * 70)
print("HISTORICAL VOLATILITY CALCULATION")
print("=" * 70)

# Download 30 trading days of historical data
# Going back more than 30 days to ensure we get 30 trading days before June 5, 2026
start_date = snapshot_date - timedelta(days=60)
end_date = snapshot_date

# Download NIFTY 50 data
try:
    nifty_data = yf.download('^NSEI', start=start_date, end=end_date, progress=False)
    print(f"\n✓ Downloaded {len(nifty_data)} days of NIFTY data")
    print(f"  Date range: {nifty_data.index[0].strftime('%Y-%m-%d')} to {nifty_data.index[-1].strftime('%Y-%m-%d')}")
except Exception as e:
    print(f"✗ Error downloading data: {e}")
    print("Using fallback: assuming historical volatility = market IV")
    nifty_data = None

# Calculate historical volatility
if nifty_data is not None and len(nifty_data) >= 30:
    # Get last 30 days
    recent_data = nifty_data.tail(30).copy()
    
    # Calculate daily log returns
    recent_data['Log_Returns'] = np.log(recent_data['Close'] / recent_data['Close'].shift(1))
    
    # Remove NaN values
    log_returns = recent_data['Log_Returns'].dropna()
    
    # Calculate daily volatility (standard deviation)
    daily_vol = log_returns.std()
    
    # Annualize volatility (multiply by sqrt(252) trading days)
    sigma_hist = daily_vol * np.sqrt(252)
    
    print(f"\n30-Day Historical Volatility Calculation:")
    print(f"  Daily volatility:    {daily_vol:.6f}")
    print(f"  Annualized σ (hist): {sigma_hist:.4f} ({sigma_hist*100:.2f}%)")
    
else:
    # Fallback: use average market IV
    avg_market_iv = (CALL_IV_MKT + PUT_IV_MKT) / 2
    sigma_hist = avg_market_iv
    print(f"\nUsing average market IV as fallback volatility: {sigma_hist*100:.2f}%")

# Use market IV for Black-Scholes (CALL_IV_MKT is already in decimal form from division by 100)
sigma = CALL_IV_MKT  # Already converted to decimal earlier

print(f"\n{'Volatility Metric':<30} {'Value':<15} {'Percentage':<15}")
print("-" * 60)
print(f"{'Historical Volatility (σ_hist)':<30} {sigma_hist:<15.6f} {sigma_hist*100:<15.2f}%")
print(f"{'Market Call IV (σ_mkt)':<30} {sigma:<15.6f} {sigma*100:<15.2f}%")

if sigma_hist > 0:
    diff_pct = ((sigma - sigma_hist) / sigma_hist) * 100
    print(f"{'Difference (σ_mkt - σ_hist)':<30} {sigma - sigma_hist:<15.6f} {diff_pct:<15.2f}%")


HISTORICAL VOLATILITY CALCULATION

✓ Downloaded 41 days of NIFTY data
  Date range: 2026-04-06 to 2026-06-04

30-Day Historical Volatility Calculation:
  Daily volatility:    0.008015
  Annualized σ (hist): 0.1272 (12.72%)

Volatility Metric              Value           Percentage     
------------------------------------------------------------
Historical Volatility (σ_hist) 0.127239        12.72          %
Market Call IV (σ_mkt)         0.142400        14.24          %
Difference (σ_mkt - σ_hist)    0.015161        11.92          %


In [10]:
# ============================================================================
# 4. BLACK-SCHOLES PRICING & GREEKS
# ============================================================================

def black_scholes_call(S, K, T, r, sigma):
    """Calculate Black-Scholes call price"""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price, d1, d2

def black_scholes_put(S, K, T, r, sigma):
    """Calculate Black-Scholes put price"""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    put_price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return put_price, d1, d2

def calculate_greeks(S, K, T, r, sigma, option_type='call'):
    """Calculate option Greeks"""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    # Delta
    if option_type == 'call':
        delta = norm.cdf(d1)
    else:
        delta = norm.cdf(d1) - 1
    
    # Gamma (same for both call and put)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    
    # Vega (same for both, per 1% change in volatility)
    vega = S * norm.pdf(d1) * np.sqrt(T) / 100  # Per 1% change
    
    # Theta (per day, so divide by 365)
    if option_type == 'call':
        theta = (-S * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) - 
                 r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
    else:
        theta = (-S * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) + 
                 r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365
    
    return {'delta': delta, 'gamma': gamma, 'vega': vega, 'theta': theta}

print("\n" + "=" * 70)
print("BLACK-SCHOLES PRICING")
print("=" * 70)

# Calculate d1 and d2
d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)

# Calculate Normal CDF values
N_d1 = norm.cdf(d1)
N_d2 = norm.cdf(d2)

# Calculate option prices
call_price, _, _ = black_scholes_call(S, K, T, r, sigma)
put_price, _, _ = black_scholes_put(S, K, T, r, sigma)

print(f"\nInputs:")
print(f"  Spot Price (S):           {S:,.2f}")
print(f"  Strike Price (K):         {K:,.2f}")
print(f"  Time to Expiry (T):       {T:.6f} years ({T_days} days)")
print(f"  Risk-Free Rate (r):       {r:.4f} ({r*100:.2f}%)")
print(f"  Volatility (σ):           {sigma:.4f} ({sigma*100:.2f}%)")

print(f"\nIntermediate Calculations:")
print(f"  d₁ = ln(S/K)/σ√T + (r + σ²/2)√T")
print(f"  d₁ = {d1:.6f}")
print(f"\n  d₂ = d₁ - σ√T")
print(f"  d₂ = {d2:.6f}")
print(f"\n  N(d₁) = {N_d1:.6f}")
print(f"  N(d₂) = {N_d2:.6f}")

print(f"\nBlack-Scholes Prices:")
print(f"  Theoretical Call Price:   {call_price:,.2f}")
print(f"  Market Call LTP:          {CALL_LTP_MKT:,.2f}")
print(f"  Error (Theo - Market):    {call_price - CALL_LTP_MKT:,.2f} ({(call_price - CALL_LTP_MKT)/CALL_LTP_MKT*100:.2f}%)")

print(f"\n  Theoretical Put Price:    {put_price:,.2f}")
print(f"  Market Put LTP:           {PUT_LTP_MKT:,.2f}")
print(f"  Error (Theo - Market):    {put_price - PUT_LTP_MKT:,.2f} ({(put_price - PUT_LTP_MKT)/PUT_LTP_MKT*100:.2f}%)")

# Store for later use
BS_CALL_PRICE = call_price
BS_PUT_PRICE = put_price
BS_D1 = d1
BS_D2 = d2
BS_ND1 = N_d1
BS_ND2 = N_d2


BLACK-SCHOLES PRICING

Inputs:
  Spot Price (S):           23,366.70
  Strike Price (K):         23,350.00
  Time to Expiry (T):       0.010959 years (4 days)
  Risk-Free Rate (r):       0.0650 (6.50%)
  Volatility (σ):           0.1424 (14.24%)

Intermediate Calculations:
  d₁ = ln(S/K)/σ√T + (r + σ²/2)√T
  d₁ = 0.103198

  d₂ = d₁ - σ√T
  d₂ = 0.088291

  N(d₁) = 0.541097
  N(d₂) = 0.535177

Black-Scholes Prices:
  Theoretical Call Price:   156.16
  Market Call LTP:          161.00
  Error (Theo - Market):    -4.84 (-3.00%)

  Theoretical Put Price:    122.84
  Market Put LTP:           105.70
  Error (Theo - Market):    17.14 (16.21%)


In [11]:
# ============================================================================
# 5. GREEKS & SCENARIO ANALYSIS
# ============================================================================

print("\n" + "=" * 70)
print("GREEKS - CALL OPTION SENSITIVITY")
print("=" * 70)

# Calculate Greeks for Call option
call_greeks = calculate_greeks(S, K, T, r, sigma, option_type='call')

delta = call_greeks['delta']
gamma = call_greeks['gamma']
vega = call_greeks['vega']
theta = call_greeks['theta']

print(f"\nGreeks for Call Option (Strike = {K:,.0f}):")
print(f"\n  Delta (∂C/∂S):     {delta:.6f}")
print(f"    → Call price increases by {delta:.4f} for each 1 point increase in spot")

print(f"\n  Gamma (∂²C/∂S²):   {gamma:.8f}")
print(f"    → Delta changes by {gamma:.8f} for each 1 point change in spot")

print(f"\n  Vega (∂C/∂σ):      {vega:.6f} per 1% vol change")
print(f"    → Call price changes by {vega:.4f} for each 1% increase in volatility")

print(f"\n  Theta (∂C/∂t):     {theta:.6f} per day")
print(f"    → Call price decreases by {abs(theta):.4f} per day (time decay)")

# Store Greeks for scenario analysis
CALL_DELTA = delta
CALL_GAMMA = gamma
CALL_VEGA = vega
CALL_THETA = theta

print("\n" + "=" * 70)
print("SCENARIO ANALYSIS: DELTA-GAMMA-VEGA APPROXIMATION")
print("=" * 70)

# Scenario: Spot increases by 1% and Volatility increases by 1% (0.01 absolute)
spot_change_pct = 0.01  # 1% increase
spot_change_points = S * spot_change_pct
vol_change_abs = 0.01  # 1% (0.01) increase in volatility

S_new = S + spot_change_points
sigma_new = sigma + vol_change_abs

print(f"\nScenario Assumptions:")
print(f"  Current Spot:              {S:,.2f}")
print(f"  New Spot (+1%):            {S_new:,.2f}")
print(f"  Spot Change (points):      +{spot_change_points:,.2f}")

print(f"\n  Current Volatility:        {sigma*100:.2f}%")
print(f"  New Volatility (+1% abs):  {sigma_new*100:.2f}%")
print(f"  Volatility Change:         +{vol_change_abs*100:.2f}%")

# Price change using Greeks approximation
# ΔC ≈ Δ * ΔS + 0.5 * Γ * (ΔS)² + Vega * Δσ

delta_price_change = delta * spot_change_points
gamma_price_change = 0.5 * gamma * (spot_change_points ** 2)
vega_price_change = vega * vol_change_abs

total_price_change = delta_price_change + gamma_price_change + vega_price_change
new_call_price_est = BS_CALL_PRICE + total_price_change

print(f"\nPrice Change Components (Greeks Approximation):")
print(f"  Delta Impact:              {delta_price_change:+,.2f}")
print(f"  Gamma Impact:              {gamma_price_change:+,.2f}")
print(f"  Vega Impact:               {vega_price_change:+,.2f}")
print(f"  {'─' * 40}")
print(f"  Total Change (approx):     {total_price_change:+,.2f}")

print(f"\nCall Option Price Projection:")
print(f"  Current Price:             {BS_CALL_PRICE:,.2f}")
print(f"  Estimated New Price:       {new_call_price_est:,.2f}")
print(f"  Estimated Change:          {total_price_change:+,.2f} ({total_price_change/BS_CALL_PRICE*100:+.2f}%)")

# For comparison: recalculate with new spot and volatility
call_price_exact, _, _ = black_scholes_call(S_new, K, T, r, sigma_new)
print(f"\nCall Option Price (Exact Recalculation with new S & σ):")
print(f"  Exact New Price:           {call_price_exact:,.2f}")
print(f"  Exact Change:              {call_price_exact - BS_CALL_PRICE:+,.2f}")
print(f"  Greeks Approximation Error: {abs(new_call_price_est - call_price_exact):,.2f} ({abs(new_call_price_est - call_price_exact)/call_price_exact*100:.2f}%)")


GREEKS - CALL OPTION SENSITIVITY

Greeks for Call Option (Strike = 23,350):

  Delta (∂C/∂S):     0.541097
    → Call price increases by 0.5411 for each 1 point increase in spot

  Gamma (∂²C/∂S²):   0.00113922
    → Delta changes by 0.00113922 for each 1 point change in spot

  Vega (∂C/∂σ):      9.706852 per 1% vol change
    → Call price changes by 9.7069 for each 1% increase in volatility

  Theta (∂C/∂t):     -19.501997 per day
    → Call price decreases by 19.5020 per day (time decay)

SCENARIO ANALYSIS: DELTA-GAMMA-VEGA APPROXIMATION

Scenario Assumptions:
  Current Spot:              23,366.70
  New Spot (+1%):            23,600.37
  Spot Change (points):      +233.67

  Current Volatility:        14.24%
  New Volatility (+1% abs):  15.24%
  Volatility Change:         +1.00%

Price Change Components (Greeks Approximation):
  Delta Impact:              +126.44
  Gamma Impact:              +31.10
  Vega Impact:               +0.10
  ────────────────────────────────────────
  Tot

In [12]:
# ============================================================================
# 6. GENERATE MARKDOWN OUTPUT
# ============================================================================

output_dir = r"c:\Users\soman\OneDrive\Coding\SOC 2026"
output_file = os.path.join(output_dir, "analysis_results.md")

markdown_content = f"""# Black-Scholes Option Pricing Analysis
## NIFTY 50 Option Chain - June 9, 2026 Expiry

**Analysis Date:** {snapshot_date.strftime('%B %d, %Y')}  
**Report Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

---

## Section 1: Parameters

| Parameter | Value |
|-----------|-------|
| Spot Price (S) | {S:,.2f} |
| Selected Strike (K) | {K:,.2f} |
| Distance from ATM | {abs(K - S):,.2f} points |
| Snapshot Date | {snapshot_date.strftime('%Y-%m-%d')} |
| Expiry Date | {expiry_date.strftime('%Y-%m-%d')} |
| Time to Maturity (T) | {T_days} days ({T:.6f} years) |
| Risk-Free Rate (r) | {r*100:.2f}% |

---

## Section 2: Volatility Comparison

### Historical vs. Market Implied Volatility

| Metric | Value | Percentage |
|--------|-------|-----------|
| **30-Day Historical Volatility (σ_hist)** | {sigma_hist:.6f} | {sigma_hist*100:.2f}% |
| **Market Call IV (σ_mkt)** | {sigma:.6f} | {sigma*100:.2f}% |
| **Difference (σ_mkt - σ_hist)** | {sigma - sigma_hist:.6f} | {((sigma - sigma_hist) / sigma_hist) * 100:+.2f}% |

**Interpretation:** The market is pricing in volatility approximately **11.92% higher** than the realized 30-day historical volatility. This suggests market expectations for elevated price movements near the expiry.

---

## Section 3: Black-Scholes Mathematical Framework

### Intermediate Calculations

The Black-Scholes model uses the following intermediate variables:

$$d_1 = \\frac{{\\ln(S/K) + (r + \\frac{{\\sigma^2}}{{2}})T}}{{\\sigma\\sqrt{{T}}}}$$

$$d_2 = d_1 - \\sigma\\sqrt{{T}}$$

**Calculated Values:**

| Variable | Formula | Value |
|----------|---------|-------|
| **d₁** | As above | {BS_D1:.6f} |
| **d₂** | d₁ - σ√T | {BS_D2:.6f} |
| **N(d₁)** | CDF of Standard Normal(d₁) | {BS_ND1:.6f} |
| **N(d₂)** | CDF of Standard Normal(d₂) | {BS_ND2:.6f} |

### Black-Scholes Option Prices

**Call Option:**

$$C = S \\cdot N(d_1) - K \\cdot e^{{-rT}} \\cdot N(d_2)$$
$$C = {S:.2f} \\times {BS_ND1:.6f} - {K:.2f} \\times e^{{-{r}\\times{T:.6f}}} \\times {BS_ND2:.6f}$$
$$C = \\boxed{{{BS_CALL_PRICE:,.2f}}}$$

**Put Option:**

$$P = K \\cdot e^{{-rT}} \\cdot N(-d_2) - S \\cdot N(-d_1)$$
$$P = {K:.2f} \\times e^{{-{r}\\times{T:.6f}}} \\times N(-{BS_D2:.6f}) - {S:.2f} \\times N(-{BS_D1:.6f})$$
$$P = \\boxed{{{BS_PUT_PRICE:,.2f}}}$$

---

## Section 4: Market vs. Theoretical Prices

### Comparison Table

| Option Type | Theoretical Price | Market LTP | Absolute Error | Percentage Error |
|-------------|-------------------|------------|-----------------|------------------|
| **Call** | {BS_CALL_PRICE:,.2f} | {CALL_LTP_MKT:,.2f} | {BS_CALL_PRICE - CALL_LTP_MKT:,.2f} | {(BS_CALL_PRICE - CALL_LTP_MKT)/CALL_LTP_MKT*100:+.2f}% |
| **Put** | {BS_PUT_PRICE:,.2f} | {PUT_LTP_MKT:,.2f} | {BS_PUT_PRICE - PUT_LTP_MKT:,.2f} | {(BS_PUT_PRICE - PUT_LTP_MKT)/PUT_LTP_MKT*100:+.2f}% |

**Analysis:**
- The **Call option** theoretical price is **{abs(BS_CALL_PRICE - CALL_LTP_MKT):.2f} points lower** than market LTP, suggesting the market is valuing calls slightly higher than Black-Scholes (possibility of upside bias or skew effects).
- The **Put option** theoretical price is **{BS_PUT_PRICE - PUT_LTP_MKT:.2f} points higher** than market LTP, indicating the market is underpricing puts relative to the model.

---

## Section 5: Greeks & Price Sensitivity

### Option Greeks (Call Option)

The Greeks represent the sensitivity of the option price to various factors:

| Greek | Symbol | Value | Interpretation |
|-------|--------|-------|-----------------|
| **Delta** | Δ | {CALL_DELTA:.6f} | Price increases {CALL_DELTA:.4f} for each 1-point move in spot |
| **Gamma** | Γ | {CALL_GAMMA:.8f} | Delta changes by {CALL_GAMMA:.8f} for each 1-point move |
| **Vega** | ν | {CALL_VEGA:.6f} | Price changes {CALL_VEGA:.4f} per 1% change in volatility |
| **Theta** | θ | {CALL_THETA:.6f}/day | Price decays {abs(CALL_THETA):.4f} per day (time decay) |

### Scenario Analysis: +1% Spot, +1% Volatility

**Scenario Setup:**
- Current Spot Price: {S:,.2f}
- New Spot Price (+1%): {S_new:,.2f} (change: +{spot_change_points:,.2f} points)
- Current Volatility: {sigma*100:.2f}%
- New Volatility: {sigma_new*100:.2f}% (change: +{vol_change_abs*100:.2f}%)

**Price Change Breakdown (Greeks Approximation):**

$$\\Delta C \\approx \\Delta \\times \\Delta S + \\frac{{1}}{{2}} \\times \\Gamma \\times (\\Delta S)^2 + \\nu \\times \\Delta\\sigma$$

| Component | Amount | Impact |
|-----------|--------|--------|
| Delta Impact | {delta_price_change:+,.2f} | +{spot_change_points:,.2f} points × {CALL_DELTA:.6f} |
| Gamma Impact | {gamma_price_change:+,.2f} | 0.5 × {CALL_GAMMA:.8f} × ({spot_change_points:,.2f})² |
| Vega Impact | {vega_price_change:+,.2f} | {CALL_VEGA:.6f} × {vol_change_abs:.4f} |
| **Total Change (Approx)** | **{total_price_change:+,.2f}** | **{total_price_change/BS_CALL_PRICE*100:+.2f}%** |

**Price Projection:**

| Metric | Value |
|--------|-------|
| Current Theoretical Call Price | {BS_CALL_PRICE:,.2f} |
| Estimated New Price (Greeks Approx) | {new_call_price_est:,.2f} |
| Exact New Price (Recalculation) | {call_price_exact:,.2f} |
| Greeks Approximation Error | {abs(new_call_price_est - call_price_exact):,.2f} ({abs(new_call_price_est - call_price_exact)/call_price_exact*100:.2f}%) |

**Conclusion:** In this scenario, the call option is expected to gain approximately **{total_price_change:,.2f} points (or {total_price_change/BS_CALL_PRICE*100:.2f}%)** in value. The Greeks approximation is highly accurate (within 1.72% error), validating the model's sensitivity measures.

---

## Appendix: Python Implementation

### Complete Analysis Script

```python
import pandas as pd
import numpy as np
from scipy.stats import norm
import yfinance as yf
from datetime import datetime, timedelta
import csv
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# BLACK-SCHOLES OPTION PRICING FRAMEWORK
# ============================================================================

def black_scholes_call(S, K, T, r, sigma):
    \"\"\"Calculate Black-Scholes call price\"\"\"
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price, d1, d2

def black_scholes_put(S, K, T, r, sigma):
    \"\"\"Calculate Black-Scholes put price\"\"\"
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    put_price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return put_price, d1, d2

def calculate_greeks(S, K, T, r, sigma, option_type='call'):
    \"\"\"Calculate option Greeks\"\"\"
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    # Delta
    if option_type == 'call':
        delta = norm.cdf(d1)
    else:
        delta = norm.cdf(d1) - 1
    
    # Gamma (same for both call and put)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    
    # Vega (same for both, per 1% change in volatility)
    vega = S * norm.pdf(d1) * np.sqrt(T) / 100  # Per 1% change
    
    # Theta (per day, so divide by 365)
    if option_type == 'call':
        theta = (-S * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) - 
                 r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
    else:
        theta = (-S * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) + 
                 r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365
    
    return {{'delta': delta, 'gamma': gamma, 'vega': vega, 'theta': theta}}

# ============================================================================
# PARAMETERS
# ============================================================================

S = {S}                    # Spot Price
K = {K}                    # Strike Price
T = {T:.6f}               # Time to Maturity
r = {r}                    # Risk-Free Rate
sigma = {sigma:.6f}       # Volatility (Market IV)

# ============================================================================
# BLACK-SCHOLES PRICING
# ============================================================================

call_price, d1, d2 = black_scholes_call(S, K, T, r, sigma)
put_price, _, _ = black_scholes_put(S, K, T, r, sigma)

print(f"Call Price: {{call_price:,.2f}}")
print(f"Put Price: {{put_price:,.2f}}")
print(f"d1: {{d1:.6f}}")
print(f"d2: {{d2:.6f}}")

# ============================================================================
# GREEKS CALCULATION
# ============================================================================

call_greeks = calculate_greeks(S, K, T, r, sigma, option_type='call')
print(f"\\nCall Delta: {{call_greeks['delta']:.6f}}")
print(f"Call Gamma: {{call_greeks['gamma']:.8f}}")
print(f"Call Vega: {{call_greeks['vega']:.6f}}")
print(f"Call Theta: {{call_greeks['theta']:.6f}}")

# ============================================================================
# SCENARIO ANALYSIS
# ============================================================================

spot_change_pct = 0.01
vol_change_abs = 0.01
S_new = S * (1 + spot_change_pct)
sigma_new = sigma + vol_change_abs

call_price_new, _, _ = black_scholes_call(S_new, K, T, r, sigma_new)
print(f"\\nNew Call Price (S +1%, σ +1%): {{call_price_new:,.2f}}")
print(f"Change: {{call_price_new - call_price:+,.2f}}")
```

---

## Summary

This analysis demonstrates the application of the Black-Scholes model to value NIFTY 50 options expiring on June 9, 2026. Key findings:

1. **Strike Selection:** The ATM strike (23,350) is 16.70 points below the spot price of 23,366.70.

2. **Volatility Regime:** Market IV (14.24%) is 11.92% higher than historical volatility (12.72%), indicating elevated market expectations.

3. **Model Pricing vs. Market:**
   - Theoretical Call: {BS_CALL_PRICE:,.2f} vs Market: {CALL_LTP_MKT:,.2f}
   - Theoretical Put: {BS_PUT_PRICE:,.2f} vs Market: {PUT_LTP_MKT:,.2f}

4. **Greeks Sensitivity:** The call option exhibits moderate Delta ({CALL_DELTA:.4f}), positive Gamma ({CALL_GAMMA:.8f}), and significant negative Theta ({CALL_THETA:.2f}/day).

5. **Scenario Outcome:** A +1% move in spot combined with +1% volatility increase would increase the call value by approximately {total_price_change:,.2f} points ({total_price_change/BS_CALL_PRICE*100:.2f}%).

---

*Analysis completed on {datetime.now().strftime('%Y-%m-%d at %H:%M:%S')} using Python Black-Scholes Framework*
"""

# Write to file
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(markdown_content)

print("\n" + "=" * 70)
print("✓ MARKDOWN OUTPUT GENERATED")
print("=" * 70)
print(f"\nOutput file saved to:")
print(f"  {output_file}")
print(f"\nFile size: {os.path.getsize(output_file):,} bytes")
print(f"\nAnalysis complete!")


✓ MARKDOWN OUTPUT GENERATED

Output file saved to:
  c:\Users\soman\OneDrive\Coding\SOC 2026\analysis_results.md

File size: 9,394 bytes

Analysis complete!
